In [1]:
from pymdp.envs import GridWorldEnv
from pymdp.jax.task import PyMDPEnv
from pymdp.jax.agent import Agent as AIFAgent


In [2]:
import jax.numpy as jnp
import jax.tree_util as jtu
from jax import nn, vmap, lax, jit
from jax import random as jr
import numpy as np

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
NUM_STATES = 4
NUM_OBS = 4
NUM_ACTIONS = 2

In [5]:
A = [jnp.broadcast_to(jnp.eye(NUM_OBS, NUM_STATES), (1,NUM_OBS, NUM_STATES))]
A[0].shape

(1, 4, 4)

In [6]:
B = [jnp.broadcast_to(jnp.ones((NUM_STATES,NUM_STATES,NUM_ACTIONS)) * 0.5, (1,NUM_STATES,NUM_STATES,NUM_ACTIONS))]
B[0].shape

(1, 4, 4, 2)

In [7]:
# B[0][0,:,0,0] = 0.9
# B[0][0,:,0,1] = 0.1
# B[0][0,:,3,0] = 0.1
# B[0][0,:,3,1] = 0.9
B[0] = B[0].at[0,:,0,0].set(0.9)
B[0] = B[0].at[0,:,0,1].set(0.1)
B[0] = B[0].at[0,:,3,0].set(0.1)
B[0] = B[0].at[0,:,3,1].set(0.9)
B[0]

Array([[[[0.9, 0.1],
         [0.5, 0.5],
         [0.5, 0.5],
         [0.1, 0.9]],

        [[0.9, 0.1],
         [0.5, 0.5],
         [0.5, 0.5],
         [0.1, 0.9]],

        [[0.9, 0.1],
         [0.5, 0.5],
         [0.5, 0.5],
         [0.1, 0.9]],

        [[0.9, 0.1],
         [0.5, 0.5],
         [0.5, 0.5],
         [0.1, 0.9]]]], dtype=float32)

In [8]:
C = [jnp.broadcast_to(jnp.array((3, 0, 5, 1)), (1,NUM_OBS))]
C[0].shape

(1, 4)

In [9]:
D = [jnp.ones((1,NUM_STATES)) * 0.5]
D[0].shape

(1, 4)

In [10]:
pB = [jnp.ones_like(B[0]) / NUM_STATES]
pB[0].shape

(1, 4, 4, 2)

In [11]:
agent = AIFAgent(A, B, C,D, pB=pB, E=None, pA=None,  learn_A=False, learn_B=True, learn_D=False, use_utility=True, policy_len=3)
agent

Agent(
  A=[f32[1,4,4]],
  B=[f32[1,4,4,2]],
  C=[i32[1,4]],
  D=[f32[1,4]],
  E=f32[1,8],
  gamma=weak_f32[1],
  alpha=weak_f32[1],
  qs=None,
  q_pi=None,
  inductive_threshold=weak_f32[1],
  inductive_epsilon=weak_f32[1],
  H=None,
  I=[f32[1,1,4]],
  pA=None,
  pB=[f32[1,4,4,2]],
  policies=i32[8,3,1],
  A_dependencies=[[0]],
  B_dependencies=[[0]],
  batch_size=1,
  num_iter=16,
  num_obs=[4],
  num_modalities=1,
  num_states=[4],
  num_factors=1,
  num_controls=[2],
  control_fac_idx=[0],
  policy_len=3,
  inductive_depth=1,
  use_utility=True,
  use_states_info_gain=True,
  use_param_info_gain=False,
  use_inductive=False,
  onehot_obs=False,
  action_selection='deterministic',
  sampling_mode='full',
  inference_algo='fpi',
  learn_A=False,
  learn_B=True,
  learn_C=False,
  learn_D=False,
  learn_E=False
)

In [12]:
print(agent.A[0].shape)

(1, 4, 4)


In [13]:
obs = [jnp.broadcast_to(jnp.array([3]), (1,1))]
obs[0].shape

(1, 1)

In [14]:
prior = [jnp.broadcast_to(jnp.ones(NUM_STATES) * 0.5, (1, NUM_STATES))]
prior[0].shape

(1, 4)

In [15]:
qs = agent.infer_states(obs, prior)
qs[0].shape

(1, 1, 4)

In [16]:
# qs = [jnp.broadcast_to(jnp.array([[0.5, 0.5]]), (1,1, NUM_STATES))]
# qs

In [17]:
qpi, nefe = agent.infer_policies(qs)
qpi


Array([[6.1570046e-12, 6.1570046e-12, 6.1570046e-12, 6.1570046e-12,
        2.5000024e-01, 2.5000024e-01, 2.5000024e-01, 2.4999928e-01]],      dtype=float32)

In [18]:
agent.sample_action(qpi)

Array([[1]], dtype=int32)

In [19]:
print(agent.qs)

None


In [20]:
def action_pair_to_obs(own, other):
    return own * 2 + other

action_pair_to_obs(0,0)

0

In [21]:
def rollout(rng_key, agent, num_timesteps):
    batch_size = agent.batch_size

    def step_fn(carry, x):
        rng_key = carry["rng_key"]
        ts = carry["ts"]

        def true_fn(carry):
            # Initialization logic for the first timestep (ts == 0)
            uncertain_qs = [jnp.ones((1, 1, NUM_STATES)) / NUM_STATES]
            qpi, _ = agent.infer_policies(uncertain_qs)
            action = agent.sample_action(qpi)
            empirical_prior, qs = agent.update_empirical_prior(action, uncertain_qs)
            
            # Use a dummy previous action for the very first observation
            obs_val = action_pair_to_obs(action[0][0], 1)
            observation = [jnp.broadcast_to(obs_val, (1, 1))]
            
            return uncertain_qs, action, observation, empirical_prior

        def false_fn(carry):
            # Logic for subsequent timesteps
            observation = carry["observation"]
            previous_action = carry["action"]
            empirical_prior = carry["empirical_prior"]
            
            qs = agent.infer_states(observation, empirical_prior)
            qpi, nefe = agent.infer_policies(qs)
            action = agent.sample_action(qpi)
            
            obs_val = action_pair_to_obs(action[0][0], previous_action[0][0])
            new_observation = [jnp.broadcast_to(obs_val, (1, 1))]
            
            new_empirical_prior, _ = agent.update_empirical_prior(action, qs)
            
            return qs, action, new_observation, new_empirical_prior

        qs, action, observation, empirical_prior = lax.cond(ts == 0, true_fn, false_fn, carry)

        history_slice = {
            "qs": qs[0],
            "action": action[0],
            "observation": observation[0],
        }

        carry = {
            "rng_key": rng_key,
            "action": action,
            "observation": observation,
            "empirical_prior": empirical_prior,
            "ts": ts + 1,
        }
        return carry, history_slice

    initial_carry = {
        "rng_key": jr.split(rng_key, 1)[0],
        "action": jnp.zeros((1, 1), dtype=jnp.int32),
        "observation": [jnp.zeros((1, 1), dtype=jnp.int32)],
        "empirical_prior": [jnp.zeros((1, NUM_STATES), dtype=jnp.float32)],
        "ts": 0
    }

    last, history = lax.scan(step_fn, initial_carry, None, length=num_timesteps + 1)
    return last, history

In [22]:
key = jr.PRNGKey(0)

jitted_func = jit(rollout, static_argnums=[2,])

In [573]:
x = jitted_func(key, agent, 50)

In [574]:
x[1]

{'action': Array([[1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1]], dtype=int32),
 'observation': Array([[[3]],
 
        [[1]],
 
        [[2]],
 
        [[3]],
 
        [[1]],
 
        [[2]],
 
        [[3]],
 
        [[1]],
 
        [[2]],
 
        [[3]],
 
        [[1]],
 
        [[2]],
 
        [[3]],
 
        [[1]],
 
        [[2]],
 
        [[3]],
 
        [[1]],
 
        

In [575]:
actions = x[1]["action"]
actions = actions[1:]
actions = jnp.broadcast_to(actions, (1, 50, 1))
actions.shape, actions


((1, 50, 1),
 Array([[[0],
         [1],
         [1],
         [0],
         [1],
         [1],
         [0],
         [1],
         [1],
         [0],
         [1],
         [1],
         [0],
         [1],
         [1],
         [0],
         [1],
         [1],
         [0],
         [1],
         [1],
         [0],
         [1],
         [1],
         [0],
         [1],
         [1],
         [0],
         [1],
         [1],
         [0],
         [1],
         [1],
         [0],
         [1],
         [1],
         [0],
         [1],
         [1],
         [0],
         [1],
         [1],
         [0],
         [1],
         [1],
         [0],
         [1],
         [1],
         [0],
         [1]]], dtype=int32))

In [576]:
observations = x[1]["observation"]
observations.shape

(51, 1, 1)

In [577]:
obs = []
for observation in observations:
    obs.append(observation[0, 0])
obs = jnp.array([obs])
obs.shape

(1, 51)

In [578]:
qs = x[1]["qs"]
qs_list = []
for q in qs:
    q = q.squeeze(0)
    q = q.squeeze(0)
    qs_list.append(q)

qs = jnp.array(qs_list)
qs = jnp.broadcast_to(qs, (1, 51, NUM_STATES))
qs.shape

(1, 51, 4)

In [579]:
[qs][0].shape[1], actions.shape[1] + 1

(51, 51)

In [580]:
agent_new = agent.infer_parameters([qs], obs, actions)

In [581]:
agent = agent_new

In [582]:
agent_new.B

[Array([[[[8.2127661e-01, 3.2894737e-03],
          [1.0638298e-01, 1.3026981e-01],
          [4.2034467e-04, 6.2500000e-02],
          [2.9850747e-03, 2.2007042e-04]],
 
         [[2.9787235e-02, 3.2894737e-03],
          [1.4893617e-01, 4.2158517e-04],
          [2.9424129e-03, 6.2500000e-02],
          [8.9253730e-01, 4.7909331e-01]],
 
         [[4.2553190e-03, 9.5065790e-01],
          [2.1276595e-02, 8.6382800e-01],
          [4.2034467e-04, 6.2500000e-02],
          [2.9850747e-03, 2.2007042e-04]],
 
         [[1.4468086e-01, 4.2763159e-02],
          [7.2340423e-01, 5.4806070e-03],
          [9.9621689e-01, 8.1250000e-01],
          [1.0149254e-01, 5.2046657e-01]]]], dtype=float32)]

In [583]:
print(f"P(CC| CC, C) = {agent_new.B[0][0, 0, 0, 0]}")
print(f"P(CC| CC, D) = {agent_new.B[0][0, 0, 0, 1]}")
print(f"P(CC| CD, C) = {agent_new.B[0][0, 0, 1, 0]}")
print(f"P(CC| CD, D) = {agent_new.B[0][0, 0, 1, 1]}")
print(f"P(CC| DC, C) = {agent_new.B[0][0, 0, 2, 0]}")
print(f"P(CC| DC, D) = {agent_new.B[0][0, 0, 2, 1]}")
print(f"P(CC| DD, C) = {agent_new.B[0][0, 0, 3, 0]}")
print(f"P(CC| DD, D) = {agent_new.B[0][0, 0, 3, 1]}")

print(f"P(CD| CC, C) = {agent_new.B[0][0, 1, 0, 0]}")
print(f"P(CD| CC, D) = {agent_new.B[0][0, 1, 0, 1]}")
print(f"P(CD| CD, C) = {agent_new.B[0][0, 1, 1, 0]}")
print(f"P(CD| CD, D) = {agent_new.B[0][0, 1, 1, 1]}")
print(f"P(CD| DC, C) = {agent_new.B[0][0, 1, 2, 0]}")
print(f"P(CD| DC, D) = {agent_new.B[0][0, 1, 2, 1]}")
print(f"P(CD| DD, C) = {agent_new.B[0][0, 1, 3, 0]}")
print(f"P(CD| DD, D) = {agent_new.B[0][0, 1, 3, 1]}")

print(f"P(DC| CC, C) = {agent_new.B[0][0, 2, 0, 0]}")
print(f"P(DC| CC, D) = {agent_new.B[0][0, 2, 0, 1]}")
print(f"P(DC| CD, C) = {agent_new.B[0][0, 2, 1, 0]}")
print(f"P(DC| CD, D) = {agent_new.B[0][0, 2, 1, 1]}")
print(f"P(DC| DC, C) = {agent_new.B[0][0, 2, 2, 0]}")
print(f"P(DC| DC, D) = {agent_new.B[0][0, 2, 2, 1]}")
print(f"P(DC| DD, C) = {agent_new.B[0][0, 2, 3, 0]}")
print(f"P(DC| DD, D) = {agent_new.B[0][0, 2, 3, 1]}")

print(f"P(DD| CC, C) = {agent_new.B[0][0, 3, 0, 0]}")
print(f"P(DD| CC, D) = {agent_new.B[0][0, 3, 0, 1]}")
print(f"P(DD| CD, C) = {agent_new.B[0][0, 3, 1, 0]}")
print(f"P(DD| CD, D) = {agent_new.B[0][0, 3, 1, 1]}")
print(f"P(DD| DC, C) = {agent_new.B[0][0, 3, 2, 0]}")
print(f"P(DD| DC, D) = {agent_new.B[0][0, 3, 2, 1]}")
print(f"P(DD| DD, C) = {agent_new.B[0][0, 3, 3, 0]}")
print(f"P(DD| DD, D) = {agent_new.B[0][0, 3, 3, 1]}")







P(CC| CC, C) = 0.8212766051292419
P(CC| CC, D) = 0.003289473708719015
P(CC| CD, C) = 0.10638298094272614
P(CC| CD, D) = 0.13026981055736542
P(CC| DC, C) = 0.0004203446733299643
P(CC| DC, D) = 0.0625
P(CC| DD, C) = 0.0029850746504962444
P(CC| DD, D) = 0.00022007041843608022
P(CD| CC, C) = 0.029787234961986542
P(CD| CC, D) = 0.003289473708719015
P(CD| CD, C) = 0.1489361673593521
P(CD| CD, D) = 0.00042158516589552164
P(CD| DC, C) = 0.0029424128588289022
P(CD| DC, D) = 0.0625
P(CD| DD, C) = 0.8925372958183289
P(CD| DD, D) = 0.4790933132171631
P(DC| CC, C) = 0.0042553190141916275
P(DC| CC, D) = 0.9506579041481018
P(DC| CD, C) = 0.021276595070958138
P(DC| CD, D) = 0.8638280034065247
P(DC| DC, C) = 0.0004203446733299643
P(DC| DC, D) = 0.0625
P(DC| DD, C) = 0.0029850746504962444
P(DC| DD, D) = 0.00022007041843608022
P(DD| CC, C) = 0.14468085765838623
P(DD| CC, D) = 0.042763158679008484
P(DD| CD, C) = 0.7234042286872864
P(DD| CD, D) = 0.0054806070402264595
P(DD| DC, C) = 0.996216893196106
P(DD|